In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS shopsphere.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE shopsphere.gold.customer_behavior AS
WITH clickstream_metrics AS (
  SELECT
    CAST(customer_id AS INT) AS customer_id,
    COUNT(DISTINCT session_id) AS sessions,
    COUNT(CASE WHEN event_type = 'Product_view' THEN 1 END) AS product_views,
    COUNT(CASE WHEN event_type = 'Add_to_cart' THEN 1 END) AS add_to_cart_events,
    COUNT(CASE WHEN event_type = 'Checkout' THEN 1 END) AS checkout_events
  FROM shopsphere.silver.clickstream
  GROUP BY CAST(customer_id AS INT)
),
order_metrics AS (
  SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS purchases
  FROM shopsphere.silver.orders
  WHERE order_status IN ('Completed', 'Delivered') -- Assuming these are successful order statuses
  GROUP BY customer_id
)
SELECT
  c.customer_id,
  c.first_name,
  c.last_name,
  c.email,
  c.city,
  c.state,
  c.country,
  c.customer_status,
  COALESCE(cm.sessions, 0) AS sessions,
  COALESCE(cm.product_views, 0) AS product_views,
  COALESCE(cm.add_to_cart_events, 0) AS add_to_cart_events,
  COALESCE(cm.checkout_events, 0) AS checkout_events,
  COALESCE(om.purchases, 0) AS purchases,
  -- Conversion rate calculations with null handling
  CASE 
    WHEN cm.product_views > 0 THEN ROUND(cm.add_to_cart_events / cm.product_views, 4)
    ELSE 0 
  END AS view_to_cart_rate,
  CASE 
    WHEN cm.add_to_cart_events > 0 THEN ROUND(cm.checkout_events / cm.add_to_cart_events, 4)
    ELSE 0 
  END AS cart_to_checkout_rate,
  CASE 
    WHEN cm.checkout_events > 0 THEN ROUND(COALESCE(om.purchases, 0) / cm.checkout_events, 4)
    ELSE 0 
  END AS checkout_to_purchase_rate,
  CASE 
    WHEN cm.sessions > 0 THEN ROUND(COALESCE(om.purchases, 0) / cm.sessions, 4)
    ELSE 0 
  END AS overall_conversion_rate
FROM shopsphere.silver.customers c
LEFT JOIN clickstream_metrics cm ON c.customer_id = cm.customer_id
LEFT JOIN order_metrics om ON c.customer_id = om.customer_id;

In [0]:
%sql
-- Sample of customer behavior with all metrics and conversion rates
SELECT 
  customer_id,
  first_name,
  last_name,
  city,
  state,
  sessions,
  product_views,
  add_to_cart_events,
  checkout_events,
  purchases,
  view_to_cart_rate,
  cart_to_checkout_rate,
  checkout_to_purchase_rate,
  overall_conversion_rate
FROM shopsphere.gold.customer_behavior
WHERE sessions > 0
ORDER BY purchases DESC, overall_conversion_rate DESC
LIMIT 20;